# 01 — LLM Configuration & Judge Calls

This notebook covers:

1. Loading runtime settings from `.env.local` / `.env`
2. Building a judge client from environment-backed defaults
3. Overriding individual parameters without extra config objects
4. Making structured judge calls with a Pydantic schema
5. Proper client cleanup with `close()` / `aclose()`

Live API cells are guarded so the notebook opens cleanly without an API key.

In [1]:
from pprint import pprint

from pydantic import BaseModel, Field

from evalf.llms import build_llm
from evalf.settings import load_runtime_settings

## 1. Inspect runtime settings

`load_runtime_settings()` reads `.env.local` first, then `.env`,
and merges them with `EVALF_*` environment variables.
The API key is excluded from the dump for safety.

In [2]:
settings = load_runtime_settings()
pprint(settings.model_dump(exclude={"api_key"}, exclude_none=True))

{'concurrency': 4,
 'k': 1,
 'max_retries': 3,
 'max_tokens': 800,
 'metric_mode': 'pass@k',
 'metrics': [],
 'model': 'gemini-2.0-flash',
 'provider': 'gemini',
 'request_timeout_seconds': 60.0,
 'temperature': 0.0,
 'threshold': 0.7}


## 2. Build a judge client from `.env`

`build_llm()` picks the configured provider, model, base URL,
and API key automatically.

Supported providers:

| Provider | Example model | Default base URL |
|----------|---------------|------------------|
| `openai` | `gpt-4.1-mini` | `https://api.openai.com/v1` |
| `gemini` | `gemini-2.0-flash` | `https://generativelanguage.googleapis.com/v1beta/openai` |
| `claude` | `claude-sonnet-4` | *(requires explicit OpenAI-compatible URL)* |

In [3]:
if settings.api_key:
    try:
        llm = build_llm()
        print(f"{type(llm).__name__}: provider={llm.provider}, model={llm.model}")
        print(f"  base_url={llm.base_url}")
    except ValueError as exc:
        llm = None
        print(f"Config error: {exc}")
else:
    llm = None
    print(
        "No API key configured.\n"
        "Set EVALF_API_KEY or a provider-specific key (OPENAI_API_KEY, etc.) "
        "to run live judge calls."
    )

GeminiLLMModel: provider=gemini, model=gemini-2.0-flash
  base_url=https://generativelanguage.googleapis.com/v1beta/openai


## 3. Override only the parameters you need

Keyword arguments to `build_llm(...)` take priority over `.env`;
anything you don't pass still falls back to environment defaults.

In [4]:
override_kwargs = {
    "timeout_seconds": 30,
    "max_retries": 2,
    "temperature": 0.0,
    "max_tokens": 300,
}
print("Overrides:")
pprint(override_kwargs)

if settings.api_key:
    try:
        llm_custom = build_llm(**override_kwargs)
        print(f"\n{type(llm_custom).__name__}: model={llm_custom.model}, max_tokens={llm_custom.max_tokens}")
    except ValueError as exc:
        llm_custom = None
        print(f"Config error: {exc}")
else:
    llm_custom = None
    print("\nSkipping (no API key).")

Overrides:
{'max_retries': 2, 'max_tokens': 300, 'temperature': 0.0, 'timeout_seconds': 30}

GeminiLLMModel: model=gemini-2.0-flash, max_tokens=300


## 4. Structured judge call

Pass `output_schema` to `generate()` and `evalf` will use
OpenAI structured output first, then fall back to plain
JSON extraction if the provider doesn't support `response_format`.

In [5]:
class QuickAnswer(BaseModel):
    answer: str = Field(description="A concise answer to the user question.")
    confidence: float = Field(ge=0.0, le=1.0)


if llm is not None:
    response = llm.generate(
        system_prompt="You are a concise assistant. Return structured output only.",
        user_prompt="Question: What does FERPA protect?",
        output_schema=QuickAnswer,
        max_tokens=120,
    )
    print("Parsed output:", response.parsed_output)
    print()
    pprint(response.usage.model_dump(exclude_none=True))
else:
    print("Skipping live request (no API key).")

Parsed output: answer='FERPA protects the privacy of student education records.' confidence=0.95

{'input_tokens': 19,
 'latency_ms': 1255.2168,
 'output_tokens': 29,
 'total_tokens': 48}


## 5. Clean up

Always close clients when you're done to release connection pools.

- `close()` releases the synchronous transport.
- `aclose()` releases the asynchronous transport.
- Both are idempotent-safe and can be called more than once.

In [6]:
for client in (llm, llm_custom):
    if client is not None:
        client.close()
        print(f"Closed {type(client).__name__}")

Closed GeminiLLMModel
Closed GeminiLLMModel
